# SprayLine DB Query Functions — 說明文件

> **版本**：v1.1（2026-06-10）  
> **對應 Schema**：v5（`setup_db.sql`，`sensor_1hour` → `sensor_3min`）  
> **對應 DataPreprocess**：`spray0609.ipynb`  

---

## 目錄

1. [前置設定（DB 連線）](#1-前置設定)
2. [Ontology vs 資料庫欄位命名比對](#2-ontology-vs-資料庫欄位命名比對)
3. [DataPreprocess JSON → 資料庫寫入函式](#3-datapreprocess-json--資料庫寫入函式)
   - 3-A `insert_batch_run`
   - 3-B `insert_timeseries_to_sensor_1min`
   - 3-C `insert_batch_sensor_to_sensor_1min`
   - 3-D `insert_3min_env_to_sensor_3min`
   - 3-E `insert_alert_event`
   - 3-F `load_json_to_db`（主入口）
4. [WebServices Query 函式（全服務）](#4-webservices-query-函式)
   - 4-A TimeSeriesService
   - 4-B BatchService
   - 4-C DashboardService
   - 4-D EventRuleService
   - 4-E TroubleshootingService
5. [前端告警顯示 JSON 建構函式（非 Query）](#5-前端告警顯示-json-建構函式非-query)
6. [完整使用範例](#6-完整使用範例)

---
## 1. 前置設定

In [ ]:
import json
import psycopg2
import psycopg2.extras
import pandas as pd
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional

# ── DB 連線設定（請依實際環境修改）──
DB_CONFIG = {
    "host":     "localhost",
    "port":     5432,
    "dbname":   "sprayline",
    "user":     "postgres",
    "password": "postgres",
}

def get_conn():
    return psycopg2.connect(**DB_CONFIG)

def query_df(sql: str, params=None) -> pd.DataFrame:
    with get_conn() as conn:
        return pd.read_sql_query(sql, conn, params=params)

print("連線設定載入完成")

---
## 2. Ontology vs 資料庫欄位命名比對

### 2-A 一致項目（✅）

| Ontology signalName | DB 資料表 | DB 欄位 | DataPreprocess JSON | 說明 |
|---|---|---|---|---|
| `air_pressure_bar` | sensor_1min | `air_pressure_bar` | `air_pressure_bar` | 空壓機壓力 |
| `filter_diff_pressure_bar` | sensor_1min | `filter_diff_pressure_bar` | `filter_diff_pressure_bar` | 濾網壓差（PdM 核心） |
| `paint_flow_ml_min` | sensor_1min | `paint_flow_ml_min` | `paint_flow_ml_min` | 塗料流量 |
| `path_error_mm` | sensor_1min | `path_error_mm` | `path_error_mm` | 軌跡追蹤誤差 |
| `servo_torque_load_pct` | sensor_1min | `servo_torque_load_pct` | `servo_torque_load_pct` | 伺服馬達負載（PdM 核心） |
| `spray_width_mm` | sensor_1min | `spray_width_mm` | `spray_width_mm` | 噴幅寬度 |
| `temperature_c` | sensor_3min | `temperature_c` | `temperature_c` | 環境溫度 |
| `humidity_rh` | sensor_3min | `humidity_rh` | `humidity_rh` | 環境濕度 |

### 2-B WebServices 命名不一致（❌）

| WebServices 欄位名 | Ontology 標準名 | 所在檔案 | 建議修正 |
|---|---|---|---|
| `spray_pressure_bar` | `air_pressure_bar` | `time_series_service.py`（v1） | 改為 `air_pressure_bar` |
| `spray_flow_rate_ml_min` | `paint_flow_ml_min` | `time_series_service.py`（v1） | 改為 `paint_flow_ml_min` |
| `pressure_bar` | `air_pressure_bar` | `time_series_service_v3/` | 改為 `air_pressure_bar` |
| `flow_rate_ml_min` | `paint_flow_ml_min` | `time_series_service_v3/` | 改為 `paint_flow_ml_min` |
| `in_flow_ml_min` | `filter_inflow_ml_min` | 兩版 WebServices | 對齊 DB 欄位名 |
| `out_flow_ml_min` | `filter_outflow_ml_min` | 兩版 WebServices | 對齊 DB 欄位名 |

### 2-C DB 欄位尚未建模至 Ontology（⚠️ 待補充）

| DB 資料表 | DB 欄位 | 說明 |
|---|---|---|
| sensor_1min | `film_thickness_um` | 膜厚（品質 Y 值） |
| sensor_1min | `nozzle_roll` | 噴嘴翻滾角 |
| sensor_1min | `filter_inflow_ml_min` / `filter_outflow_ml_min` | 濾網進出液流量 |
| sensor_1min | `pump_current_a` | 幫浦電流 |
| sensor_1min | `vibration_g` | 振動加速度 |
| sensor_1min | `tcp_x/y/z_mm` / `speed_mm_s` | TCP 座標與噴塗速度 |
| sensor_3min | `gearbox_temperature_c` | 減速機溫度（3 分鐘採樣） |

---
## 3. DataPreprocess JSON → 資料庫寫入函式

### DataPreprocess JSON 格式

```json
{
  "time_series": [
    {
      "timestamp":                "2026-06-09T08:00:00+08:00",
      "station":                  "Station_1",
      "temperature_c":            25.3,
      "humidity_rh":              55.0,
      "filter_diff_pressure_bar": 0.25,
      "servo_torque_load_pct":    52.0,
      "air_pressure_bar":         2.51,
      "data_quality_flag":        "normal"
    }
  ],
  "batches": [
    {
      "event_type":        "batch_completed",
      "station":           "Station_1",
      "batch_id":          "B_20260609_001",
      "batch_start_time":  "2026-06-09T08:00:00+08:00",
      "batch_end_time":    "2026-06-09T08:15:00+08:00",
      "paint_flow_ml_min": 115.2,
      "spray_width_mm":    112.0,
      "path_error_mm":     0.05,
      "data_quality_flag": "normal"
    }
  ]
}
```

### 欄位對應表（JSON → DB）

| JSON 欄位 | 目標資料表 | DB 欄位 | 備註 |
|---|---|---|---|
| `batch_id` | batch_run | `batch_id` | PK |
| `batch_start_time` | batch_run | `start_time` | |
| `batch_end_time` | batch_run | `ended_time` | |
| `timestamp` | sensor_1min | `ts` | |
| `station` | sensor_1min | `station_id` | |
| `filter_diff_pressure_bar` | sensor_1min | `filter_diff_pressure_bar` | |
| `servo_torque_load_pct` | sensor_1min | `servo_torque_load_pct` | |
| `air_pressure_bar` | sensor_1min | `air_pressure_bar` | |
| `paint_flow_ml_min` | sensor_1min | `paint_flow_ml_min` | 批次層級 |
| `spray_width_mm` | sensor_1min | `spray_width_mm` | 批次層級 |
| `path_error_mm` | sensor_1min | `path_error_mm` | 批次層級 |
| `temperature_c` | **sensor_3min** | `temperature_c` | 每 3 分鐘聚合 |
| `humidity_rh` | **sensor_3min** | `humidity_rh` | 每 3 分鐘聚合 |

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 3-A  insert_batch_run
# ═══════════════════════════════════════════════════════════════

def insert_batch_run(conn, batch: dict) -> Optional[str]:
    """
    將 batches 清單中的 batch_completed 記錄寫入 batch_run。

    Returns
    -------
    batch_id : str | None — 新寫入時回傳 batch_id，重複時回傳 None
    """
    if batch.get("event_type") != "batch_completed":
        return None
    sql = """
        INSERT INTO batch_run (batch_id, start_time, ended_time, status)
        VALUES (%(batch_id)s, %(start_time)s, %(ended_time)s, 'ok')
        ON CONFLICT (batch_id) DO NOTHING
        RETURNING batch_id;
    """
    with conn.cursor() as cur:
        cur.execute(sql, {
            "batch_id":   batch["batch_id"],
            "start_time": batch["batch_start_time"],
            "ended_time": batch["batch_end_time"],
        })
        row = cur.fetchone()
        conn.commit()
        return row[0] if row else None


# ═══════════════════════════════════════════════════════════════
# 3-B  insert_timeseries_to_sensor_1min
#       time_series 1Hz 欄位 → sensor_1min
# ═══════════════════════════════════════════════════════════════

def insert_timeseries_to_sensor_1min(
    conn,
    ts_records: List[dict],
    batch_id: str
) -> int:
    """
    批量寫入 time_series 中的 1Hz PdM 感測欄位至 sensor_1min。

    寫入欄位：filter_diff_pressure_bar, air_pressure_bar, servo_torque_load_pct

    Returns
    -------
    inserted_count : int
    """
    sql = """
        INSERT INTO sensor_1min (
            ts, batch_id, station_id,
            filter_diff_pressure_bar,
            air_pressure_bar,
            servo_torque_load_pct
        )
        VALUES (
            %(ts)s, %(batch_id)s, %(station_id)s,
            %(filter_diff_pressure_bar)s,
            %(air_pressure_bar)s,
            %(servo_torque_load_pct)s
        )
        ON CONFLICT DO NOTHING;
    """
    rows = [
        {
            "ts":                       rec["timestamp"],
            "batch_id":                 batch_id,
            "station_id":               rec["station"],
            "filter_diff_pressure_bar": rec.get("filter_diff_pressure_bar"),
            "air_pressure_bar":         rec.get("air_pressure_bar"),
            "servo_torque_load_pct":    rec.get("servo_torque_load_pct"),
        }
        for rec in ts_records
    ]
    with conn.cursor() as cur:
        psycopg2.extras.execute_batch(cur, sql, rows, page_size=500)
        conn.commit()
    return len(rows)


# ═══════════════════════════════════════════════════════════════
# 3-C  insert_batch_sensor_to_sensor_1min
#       批次層級感測平均值（flow / spray_width / path_error）
# ═══════════════════════════════════════════════════════════════

def insert_batch_sensor_to_sensor_1min(conn, batch: dict) -> bool:
    """
    將批次層級感測平均值（paint_flow_ml_min / spray_width_mm / path_error_mm）
    以 batch_end_time 為 ts 寫入 sensor_1min。

    Returns
    -------
    success : bool
    """
    if batch.get("event_type") != "batch_completed":
        return False
    sql = """
        INSERT INTO sensor_1min (
            ts, batch_id, station_id,
            paint_flow_ml_min, spray_width_mm, path_error_mm
        )
        VALUES (
            %(ts)s, %(batch_id)s, %(station_id)s,
            %(paint_flow_ml_min)s, %(spray_width_mm)s, %(path_error_mm)s
        )
        ON CONFLICT DO NOTHING;
    """
    with conn.cursor() as cur:
        cur.execute(sql, {
            "ts":               batch["batch_end_time"],
            "batch_id":         batch["batch_id"],
            "station_id":       batch["station"],
            "paint_flow_ml_min": batch.get("paint_flow_ml_min"),
            "spray_width_mm":    batch.get("spray_width_mm"),
            "path_error_mm":     batch.get("path_error_mm"),
        })
        conn.commit()
    return True


# ═══════════════════════════════════════════════════════════════
# 3-D  insert_3min_env_to_sensor_3min
#       環境資料每 3 分鐘聚合 → sensor_3min（原 sensor_1hour 已改版）
# ═══════════════════════════════════════════════════════════════

def insert_3min_env_to_sensor_3min(
    conn,
    ts_records: List[dict],
    station_id: str,
    batch_id: Optional[str] = None
) -> int:
    """
    將 time_series 中的 temperature_c / humidity_rh 依「3 分鐘取平均」
    後寫入 sensor_3min。

    Parameters
    ----------
    ts_records : list[dict] — DataPreprocess time_series 陣列
    station_id : str — e.g. 'Station_1'
    batch_id   : str | None — 選填

    Returns
    -------
    inserted_count : int
    """
    df = pd.DataFrame(ts_records)
    if df.empty:
        return 0

    df["ts"] = pd.to_datetime(df["timestamp"], utc=True)
    df = df[df["station"] == station_id].copy()
    if df.empty:
        return 0

    # 3 分鐘 floor（e.g. 08:00, 08:03, 08:06 ...）
    df["bucket"] = df["ts"].dt.floor("3min")

    cols_available = [c for c in ["temperature_c", "humidity_rh", "gearbox_temperature_c"]
                      if c in df.columns]
    agg_df = (
        df.groupby("bucket")[cols_available]
        .mean()
        .reset_index()
    )

    sql = """
        INSERT INTO sensor_3min (
            ts, batch_id, station_id,
            temperature_c, humidity_rh, gearbox_temperature_c
        )
        VALUES (
            %(ts)s, %(batch_id)s, %(station_id)s,
            %(temperature_c)s, %(humidity_rh)s, %(gearbox_temperature_c)s
        )
        ON CONFLICT DO NOTHING;
    """
    rows = [
        {
            "ts":                   row["bucket"].isoformat(),
            "batch_id":             batch_id,
            "station_id":           station_id,
            "temperature_c":        round(float(row["temperature_c"]), 2)
                                    if "temperature_c" in agg_df.columns else None,
            "humidity_rh":          round(float(row["humidity_rh"]), 2)
                                    if "humidity_rh" in agg_df.columns else None,
            "gearbox_temperature_c": round(float(row["gearbox_temperature_c"]), 2)
                                    if "gearbox_temperature_c" in agg_df.columns else None,
        }
        for _, row in agg_df.iterrows()
    ]
    with conn.cursor() as cur:
        psycopg2.extras.execute_batch(cur, sql, rows)
        conn.commit()
    return len(rows)


print("✅ 3-A～3-D 寫入函式定義完成")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 3-E  insert_alert_event
#       直接寫入單筆告警事件，並可選擇性連結 cause / response
# ═══════════════════════════════════════════════════════════════

def insert_alert_event(
    conn,
    batch_id: str,
    station_id: str,
    sensor_name: str,
    measured_value: float,
    state: str,
    cause_id: Optional[str] = None,
    response_id: Optional[str] = None,
    ts: Optional[str] = None,
    message: Optional[str] = None,
    operator_id: Optional[str] = None,
) -> dict:
    """
    寫入單筆 alert_event，並可選擇性建立 cause / response 連結。

    Parameters
    ----------
    batch_id       : str  — 對應 batch_run.batch_id（必填）
    station_id     : str  — e.g. 'Station_1'（必填）
    sensor_name    : str  — 觸發告警的感測欄位名（必填）
    measured_value : float — 觸發時的感測值（必填）
    state          : str  — 'warning' | 'fault'（必填）
    cause_id       : str | None — cause_catalog.cause_id；有值時寫 alert_cause_link
    response_id    : str | None — response_catalog.response_id；有值時寫 alert_response_link
    ts             : str | None — ISO 8601；None 時使用 DB NOW()
    message        : str | None — 補充說明
    operator_id    : str | None — 執行 response 的操作員（搭配 response_id 使用）

    Returns
    -------
    dict:
        'event_id'   : str  — 新建 alert_event 的 UUID
        'cause_linked'    : bool
        'response_linked' : bool

    Raises
    ------
    ValueError  — state 不在允許值範圍內
    """
    if state not in ("warning", "fault"):
        raise ValueError(f"state 必須為 'warning' 或 'fault'，收到：{state!r}")

    # ── 1. 寫入 alert_event ──
    sql_event = """
        INSERT INTO alert_event (
            batch_id, station_id, sensor_name, measured_value,
            state, cause, ts, message
        )
        VALUES (
            %(batch_id)s, %(station_id)s, %(sensor_name)s, %(measured_value)s,
            %(state)s, %(cause_id)s,
            COALESCE(%(ts)s::timestamptz, NOW()),
            %(message)s
        )
        RETURNING event_id;
    """

    with conn.cursor() as cur:
        cur.execute(sql_event, {
            "batch_id":       batch_id,
            "station_id":     station_id,
            "sensor_name":    sensor_name,
            "measured_value": measured_value,
            "state":          state,
            "cause_id":       cause_id,
            "ts":             ts,
            "message":        message,
        })
        event_id = str(cur.fetchone()[0])

        # ── 2. 選擇性寫入 alert_cause_link ──
        cause_linked = False
        if cause_id:
            cur.execute(
                """
                INSERT INTO alert_cause_link (alert_id, cause_id, is_primary)
                VALUES (%s, %s, TRUE)
                ON CONFLICT DO NOTHING;
                """,
                (event_id, cause_id)
            )
            cause_linked = True

        # ── 3. 選擇性寫入 alert_response_link ──
        response_linked = False
        if response_id:
            cur.execute(
                """
                INSERT INTO alert_response_link (alert_id, response_id, operator_id)
                VALUES (%s, %s, %s)
                ON CONFLICT DO NOTHING;
                """,
                (event_id, response_id, operator_id)
            )
            response_linked = True

        conn.commit()

    return {
        "event_id":        event_id,
        "cause_linked":    cause_linked,
        "response_linked": response_linked,
    }


print("✅ 3-E insert_alert_event 定義完成")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 3-F  load_json_to_db — 主入口：解包整份 DataPreprocess JSON
# ═══════════════════════════════════════════════════════════════

def load_json_to_db(conn, payload: dict) -> dict:
    """
    將 DataPreprocess 輸出的完整 JSON payload 解包並寫入各資料表。

    Parameters
    ----------
    payload : dict — { 'time_series': [...], 'batches': [...] }

    Returns
    -------
    report : dict — 各步驟寫入筆數
    """
    ts_records = payload.get("time_series", [])
    batches    = payload.get("batches", [])

    report = {
        "batch_run_inserted": 0,
        "sensor_1min_ts":     0,
        "sensor_1min_batch":  0,
        "sensor_3min_env":    0,   # 原 sensor_1hour，已改為 3 分鐘
        "skipped_batches":    [],
    }

    for batch in batches:
        batch_id = insert_batch_run(conn, batch)
        if batch_id is None:
            report["skipped_batches"].append(batch.get("batch_id"))
            batch_id = batch["batch_id"]
        else:
            report["batch_run_inserted"] += 1

        station_id = batch["station"]
        station_ts = [r for r in ts_records if r.get("station") == station_id]

        report["sensor_1min_ts"] += insert_timeseries_to_sensor_1min(
            conn, station_ts, batch_id
        )
        if insert_batch_sensor_to_sensor_1min(conn, batch):
            report["sensor_1min_batch"] += 1

        # 環境資料：每 3 分鐘聚合 → sensor_3min
        report["sensor_3min_env"] += insert_3min_env_to_sensor_3min(
            conn, station_ts, station_id, batch_id
        )

    return report


# ── 範例 payload ──
SAMPLE_PAYLOAD = {
    "time_series": [
        {
            "timestamp": "2026-06-09T08:00:00+08:00", "station": "Station_1",
            "temperature_c": 25.3, "humidity_rh": 55.0,
            "filter_diff_pressure_bar": 0.25, "servo_torque_load_pct": 52.0,
            "air_pressure_bar": 2.51, "data_quality_flag": "normal"
        },
        {
            "timestamp": "2026-06-09T08:01:00+08:00", "station": "Station_1",
            "temperature_c": 25.4, "humidity_rh": 55.2,
            "filter_diff_pressure_bar": 0.26, "servo_torque_load_pct": 52.5,
            "air_pressure_bar": 2.52, "data_quality_flag": "normal"
        },
        {
            "timestamp": "2026-06-09T08:02:00+08:00", "station": "Station_1",
            "temperature_c": 25.4, "humidity_rh": 55.1,
            "filter_diff_pressure_bar": 0.26, "servo_torque_load_pct": 53.0,
            "air_pressure_bar": 2.50, "data_quality_flag": "normal"
        },
    ],
    "batches": [
        {
            "event_type": "batch_completed", "station": "Station_1",
            "batch_id": "B_20260609_001",
            "batch_start_time": "2026-06-09T08:00:00+08:00",
            "batch_end_time":   "2026-06-09T08:15:00+08:00",
            "paint_flow_ml_min": 115.2, "spray_width_mm": 112.0,
            "path_error_mm": 0.05, "data_quality_flag": "normal"
        }
    ]
}

print("✅ 3-F load_json_to_db 定義完成")
# conn = get_conn()
# print(json.dumps(load_json_to_db(conn, SAMPLE_PAYLOAD), indent=2, ensure_ascii=False))
# conn.close()

---
## 4. WebServices Query 函式（全服務）

---
### 4-A TimeSeriesService

**Contract**：`time_series_service_contract.py`  
**DB 資料表**：`sensor_1min`、`sensor_3min`

In [ ]:
# 門檻值（依 DataPreprocess spray0609.ipynb 定義）
SENSOR_THRESHOLDS = {
    "filter_diff_pressure_bar": {"warning": 0.5,  "fault": 0.7},
    "servo_torque_load_pct":    {"warning": 60.0, "fault": 75.0},
    "air_pressure_bar":         {"warning_lo": 2.3, "warning_hi": 2.7},
    "temperature_c":            {"warning_lo": 15.0, "warning_hi": 35.0},
    "humidity_rh":              {"warning_lo": 35.0, "warning_hi": 75.0},
    "paint_flow_ml_min":        {"warning_lo": 95.0, "warning_hi": 125.0},
    "spray_width_mm":           {"warning_lo": 95.0, "warning_hi": 125.0},
    "path_error_mm":            {"warning": 0.10, "fault": 0.15},
}

ALL_SENSOR_1MIN_COLS = [
    "filter_diff_pressure_bar", "air_pressure_bar", "paint_flow_ml_min",
    "spray_width_mm", "servo_torque_load_pct", "path_error_mm",
    "film_thickness_um", "nozzle_roll", "filter_inflow_ml_min",
    "filter_outflow_ml_min", "pump_current_a", "vibration_g",
    "tcp_x_mm", "tcp_y_mm", "tcp_z_mm", "speed_mm_s",
]


def get_time_series(
    conn,
    station: str,
    start_time: str,
    end_time: str,
    sensor_names: Optional[List[str]] = None
) -> pd.DataFrame:
    """
    查詢指定站點、時間窗的 sensor_1min 資料。

    Parameters
    ----------
    station      : str  — e.g. 'Station_1'
    start_time   : str  — ISO 8601
    end_time     : str  — ISO 8601
    sensor_names : list[str] | None — 指定感測欄位；None 回傳全部
    """
    cols = sensor_names if sensor_names else ALL_SENSOR_1MIN_COLS
    valid_cols = [c for c in cols if c in ALL_SENSOR_1MIN_COLS]
    select_cols = ", ".join(f"s1m.{c}" for c in valid_cols)

    sql = f"""
        SELECT s1m.ts, s1m.station_id, s1m.batch_id, {select_cols}
        FROM sensor_1min s1m
        WHERE s1m.station_id = %(station)s
          AND s1m.ts BETWEEN %(start_time)s AND %(end_time)s
        ORDER BY s1m.ts ASC;
    """
    return pd.read_sql_query(sql, conn, params={
        "station": station, "start_time": start_time, "end_time": end_time,
    })


def get_latest_sensor_snapshot(conn, station: str) -> dict:
    """
    查詢指定站點最新一筆 sensor_1min 資料。
    """
    sql = """
        SELECT * FROM sensor_1min
        WHERE station_id = %(station)s
        ORDER BY ts DESC LIMIT 1;
    """
    with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
        cur.execute(sql, {"station": station})
        row = cur.fetchone()
        return dict(row) if row else {}


def get_sensor_feature_window(
    conn,
    station: str,
    sensor_name: str,
    start_time: str,
    end_time: str,
    aggregation_window: str = "5 minutes"
) -> pd.DataFrame:
    """
    計算感測欄位的滾動聚合特徵（avg / stddev / 超標次數）。

    Parameters
    ----------
    aggregation_window : str — PostgreSQL interval，e.g. '5 minutes', '1 hour'
                               需啟用 TimescaleDB 的 time_bucket()。
    """
    threshold = SENSOR_THRESHOLDS.get(sensor_name, {}).get("warning")
    over_expr = (
        f"SUM(CASE WHEN {sensor_name} > {threshold} THEN 1 ELSE 0 END)"
        if threshold is not None else "NULL"
    )
    sql = f"""
        SELECT
            time_bucket(%(agg)s::interval, ts) AS window_start,
            AVG({sensor_name})    AS avg,
            STDDEV({sensor_name}) AS stddev,
            COUNT(*)              AS count,
            {over_expr}           AS over_threshold_count
        FROM sensor_1min
        WHERE station_id = %(station)s
          AND ts BETWEEN %(start_time)s AND %(end_time)s
          AND {sensor_name} IS NOT NULL
        GROUP BY window_start
        ORDER BY window_start ASC;
    """
    return pd.read_sql_query(sql, conn, params={
        "station": station, "start_time": start_time,
        "end_time": end_time, "agg": aggregation_window,
    })


print("✅ 4-A TimeSeriesService 定義完成")

---
### 4-B BatchService

**Contract**：`batch_service_contract.py`  
**DB 資料表**：`batch_run`、`batch_station_status`、`alert_event`

In [ ]:
def list_batches_filtered(
    conn,
    date: str,
    risk: Optional[str] = None,
    start: Optional[str] = None,
    end: Optional[str] = None,
    station: Optional[str] = None
) -> pd.DataFrame:
    """
    查詢批次清單，支援日期、風險等級、時段、站點篩選。

    Parameters
    ----------
    date    : str — e.g. '2026-06-09'
    risk    : str | None — 'ok' / 'warning' / 'bad'
    start   : str | None — 時段起點 'HH:MM'
    end     : str | None — 時段終點 'HH:MM'
    station : str | None — e.g. 'Station_1'
    """
    sql = """
        SELECT
            br.batch_id, br.start_time, br.ended_time, br.status,
            bss.station_id,
            bss.robot_arm_state, bss.nozzle_state, bss.filter_state,
            bss.compressor_state, bss.spray_width_state, bss.quality_state
        FROM batch_run br
        LEFT JOIN batch_station_status bss ON br.batch_id = bss.batch_id
        WHERE br.start_time::date = %(date)s
          AND (%(risk)s    IS NULL OR br.status      = %(risk)s)
          AND (%(station)s IS NULL OR bss.station_id = %(station)s)
          AND (%(start)s   IS NULL OR br.start_time::time >= %(start)s::time)
          AND (%(end)s     IS NULL OR br.start_time::time <= %(end)s::time)
        ORDER BY br.start_time DESC;
    """
    return pd.read_sql_query(sql, conn, params={
        "date": date, "risk": risk, "station": station,
        "start": start, "end": end,
    })


def get_batch_detail(conn, batch_id: str) -> dict:
    """
    查詢單一批次的完整資訊（基本資料 + 站點狀態 + 告警事件）。
    """
    params = {"batch_id": batch_id}
    with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
        cur.execute("SELECT * FROM batch_run WHERE batch_id = %(batch_id)s;", params)
        batch = dict(cur.fetchone() or {})

        cur.execute(
            "SELECT * FROM batch_station_status WHERE batch_id = %(batch_id)s ORDER BY station_id;",
            params
        )
        stations = [dict(r) for r in cur.fetchall()]

        cur.execute(
            """
            SELECT event_id, station_id, sensor_name, measured_value,
                   state, cause, ts, message, acknowledged_at
            FROM alert_event WHERE batch_id = %(batch_id)s ORDER BY ts ASC;
            """,
            params
        )
        alerts = [dict(r) for r in cur.fetchall()]

    return {"batch": batch, "stations": stations, "alerts": alerts}


print("✅ 4-B BatchService 定義完成")

---
### 4-C DashboardService

**Contract**：`dashboard_service_contract.py`

In [ ]:
def get_manager_summary(conn, date: str, station: Optional[str] = None) -> dict:
    """
    管理者儀表板摘要（批次統計 + 未確認告警 + 站點分布）。

    Parameters
    ----------
    date    : str — e.g. '2026-06-09'
    station : str | None — 篩選單一站點
    """
    params = {"date": date, "station": station}

    sql_summary = """
        SELECT
            COUNT(DISTINCT br.batch_id) AS total_batches,
            COUNT(DISTINCT CASE WHEN br.status = 'ok'      THEN br.batch_id END) AS ok_batches,
            COUNT(DISTINCT CASE WHEN br.status = 'warning' THEN br.batch_id END) AS warning_batches,
            COUNT(DISTINCT CASE WHEN br.status = 'bad'     THEN br.batch_id END) AS bad_batches,
            COUNT(ae.event_id)                                                    AS total_alerts,
            COUNT(CASE WHEN ae.acknowledged_at IS NULL THEN ae.event_id END)      AS unacknowledged_alerts
        FROM batch_run br
        LEFT JOIN batch_station_status bss ON br.batch_id = bss.batch_id
        LEFT JOIN alert_event ae           ON br.batch_id = ae.batch_id
        WHERE br.start_time::date = %(date)s
          AND (%(station)s IS NULL OR bss.station_id = %(station)s);
    """
    sql_breakdown = """
        SELECT
            bss.station_id,
            COUNT(DISTINCT bss.batch_id)                                         AS batch_count,
            SUM(CASE WHEN bss.filter_state     = 'fault' THEN 1 ELSE 0 END)      AS filter_fault,
            SUM(CASE WHEN bss.robot_arm_state  = 'fault' THEN 1 ELSE 0 END)      AS robot_arm_fault,
            SUM(CASE WHEN bss.nozzle_state     = 'fault' THEN 1 ELSE 0 END)      AS nozzle_fault,
            SUM(CASE WHEN bss.compressor_state = 'fault' THEN 1 ELSE 0 END)      AS compressor_fault,
            SUM(CASE WHEN bss.quality_state    = 'fault' THEN 1 ELSE 0 END)      AS quality_fault
        FROM batch_station_status bss
        JOIN batch_run br ON bss.batch_id = br.batch_id
        WHERE br.start_time::date = %(date)s
          AND (%(station)s IS NULL OR bss.station_id = %(station)s)
        GROUP BY bss.station_id
        ORDER BY bss.station_id;
    """
    with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
        cur.execute(sql_summary, params)
        summary = dict(cur.fetchone() or {})
        cur.execute(sql_breakdown, params)
        breakdown = [dict(r) for r in cur.fetchall()]

    return {
        "date": date, "station": station,
        "total_batches":         summary.get("total_batches", 0),
        "ok_batches":            summary.get("ok_batches", 0),
        "warning_batches":       summary.get("warning_batches", 0),
        "bad_batches":           summary.get("bad_batches", 0),
        "total_alerts":          summary.get("total_alerts", 0),
        "unacknowledged_alerts": summary.get("unacknowledged_alerts", 0),
        "station_breakdown":     breakdown,
    }


print("✅ 4-C DashboardService 定義完成")

---
### 4-D EventRuleService

**Contract**：`event_rule_service_contract.py`

In [ ]:
def list_unacknowledged_alert_events(
    conn,
    station: Optional[str] = None,
    severity: Optional[str] = None
) -> pd.DataFrame:
    """
    查詢所有未確認告警（含關聯原因）。

    Parameters
    ----------
    station  : str | None
    severity : str | None — 'low' / 'medium' / 'high'（來自 cause_catalog）
    """
    sql = """
        SELECT
            ae.event_id, ae.batch_id, ae.station_id,
            ae.sensor_name, ae.measured_value, ae.state,
            ae.cause, ae.ts, ae.message,
            acl.cause_id, acl.is_primary,
            cc.description_zh AS cause_description,
            cc.severity        AS cause_severity
        FROM alert_event ae
        LEFT JOIN alert_cause_link acl ON ae.event_id = acl.alert_id
        LEFT JOIN cause_catalog    cc  ON acl.cause_id = cc.cause_id
        WHERE ae.acknowledged_at IS NULL
          AND (%(station)s  IS NULL OR ae.station_id = %(station)s)
          AND (%(severity)s IS NULL OR cc.severity   = %(severity)s)
        ORDER BY ae.ts DESC;
    """
    return pd.read_sql_query(sql, conn, params={"station": station, "severity": severity})


def acknowledge_alert_event(conn, event_id: str, operator_id: str) -> dict:
    """
    確認告警事件（寫入 acknowledged_at）。
    """
    sql = """
        UPDATE alert_event
        SET acknowledged_at = NOW()
        WHERE event_id = %(event_id)s AND acknowledged_at IS NULL
        RETURNING event_id, acknowledged_at;
    """
    with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
        cur.execute(sql, {"event_id": event_id})
        row = cur.fetchone()
        conn.commit()
        return dict(row) if row else {}


def evaluate_event_rules(
    conn, station: str, batch_id: str,
    timestamp: str, sensor_payload: dict
) -> List[dict]:
    """
    依 SENSOR_THRESHOLDS 批次評估感測值，超標時呼叫 insert_alert_event。

    Returns
    -------
    list[dict] — 已觸發的 alert 記錄（含 event_id）
    """
    triggered = []
    for sensor_name, value in sensor_payload.items():
        if value is None or sensor_name not in SENSOR_THRESHOLDS:
            continue
        thresh = SENSOR_THRESHOLDS[sensor_name]
        state = None
        if "fault" in thresh and value > thresh["fault"]:
            state = "fault"
        elif "warning" in thresh and value > thresh["warning"]:
            state = "warning"
        if state:
            result = insert_alert_event(
                conn, batch_id, station, sensor_name, value, state,
                ts=timestamp,
                message=f"{sensor_name}={value} exceeded {state} threshold"
            )
            triggered.append(result)
    return triggered


print("✅ 4-D EventRuleService 定義完成")

---
### 4-E TroubleshootingService

**Contract**：`troubleshooting_service_contract.py`

In [ ]:
def get_troubleshooting_matrix(
    conn,
    asset_type: Optional[str] = None,
    issue_type: Optional[str] = None
) -> pd.DataFrame:
    """
    查詢元件－問題－解方三維知識矩陣。

    Parameters
    ----------
    asset_type : str | None — e.g. 'FILTER' / 'ROBOT_ARM'
    issue_type : str | None — e.g. 'FILTER_CLOG'
    """
    sql = """
        SELECT
            cc.component_id, cc.display_name   AS component_name,
            ic.issue_id,     ic.display_name   AS issue_name,    ic.severity,
            sc.solution_id,  sc.description    AS solution_description,
            sc.downtime_estimate_min, sc.skill_required,
            cism.relevance_rank, cism.effectiveness_pct
        FROM component_issue_solution_map cism
        JOIN component_catalog cc ON cism.component_id = cc.component_id
        JOIN issue_catalog      ic ON cism.issue_id     = ic.issue_id
        JOIN solution_catalog   sc ON cism.solution_id  = sc.solution_id
        WHERE (%(asset_type)s IS NULL OR cc.component_id = %(asset_type)s)
          AND (%(issue_type)s IS NULL OR ic.issue_id     = %(issue_type)s)
        ORDER BY cc.component_id, ic.severity DESC, cism.relevance_rank ASC;
    """
    return pd.read_sql_query(sql, conn, params={"asset_type": asset_type, "issue_type": issue_type})


def get_issue_recommendations(conn, issue_type: str, station: Optional[str] = None) -> dict:
    """
    查詢特定問題類型的候選解方（依 relevance_rank 排序）。
    """
    with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
        cur.execute("SELECT * FROM issue_catalog WHERE issue_id = %s;", (issue_type,))
        issue = dict(cur.fetchone() or {})

        cur.execute(
            """
            SELECT sc.solution_id, sc.description AS solution,
                   sc.downtime_estimate_min, sc.skill_required,
                   cism.relevance_rank, cism.effectiveness_pct,
                   cc.component_id, cc.display_name AS component_name
            FROM component_issue_solution_map cism
            JOIN solution_catalog  sc ON cism.solution_id  = sc.solution_id
            JOIN component_catalog cc ON cism.component_id = cc.component_id
            WHERE cism.issue_id = %s
            ORDER BY cism.relevance_rank ASC;
            """,
            (issue_type,)
        )
        recs = [dict(r) for r in cur.fetchall()]

    return {"issue": issue, "recommendations": recs}


print("✅ 4-E TroubleshootingService 定義完成")

---
## 5. 前端告警顯示 JSON 建構函式（非 Query）

### 設計目的

前端需要在告警面板同時顯示：

| 顯示欄位 | 來源資料表 | 對應欄位 |
|---|---|---|
| 問題原因 | `cause_catalog` | `description_zh` |
| 建議解法 | `response_catalog` | `description_zh` |
| 負責單位 | `response_catalog` | `skill_required` → 中文標籤 |
| 停機時間預估 | `response_catalog` | `downtime_estimate_min` |
| 嚴重度 | `cause_catalog` | `severity` |

`build_alert_display_json` 為**組合型函式**（非單純 Query wrapper），
負責整合多表查詢結果並輸出前端可直接渲染的 JSON 結構。

### 輸出 JSON 結構

```json
{
  "event_id": "uuid",
  "station_id": "Station_1",
  "sensor_name": "filter_diff_pressure_bar",
  "sensor_display_name": "濾網壓差",
  "measured_value": 0.65,
  "unit": "bar",
  "state": "fault",
  "ts": "2026-06-09T08:30:00+08:00",
  "batch_id": "B_20260609_001",
  "acknowledged": false,
  "problem": {
    "summary": "Station_1 濾網壓差 偵測到 嚴重故障（測量值：0.65 bar）",
    "causes": [
      {
        "cause_id": "FILTER_CLOG",
        "description": "漆渣累積導致濾網阻塞，壓差持續上升",
        "category": "pdm_core",
        "severity": "medium",
        "is_primary": true
      }
    ]
  },
  "actions": [
    {
      "response_id": "REPLACE_FILTER",
      "description": "更換濾網（壓差超過門檻觸發）",
      "responsible_unit_code": "operator",
      "responsible_unit_zh": "作業員",
      "downtime_estimate_min": 30,
      "downtime_display": "約 30 分鐘",
      "is_executed": false,
      "executed_at": null,
      "executed_by": null
    }
  ],
  "generated_at": "2026-06-10T08:35:00+08:00"
}
```

In [ ]:
# ── 感測欄位顯示名稱與單位對照表 ──
SENSOR_DISPLAY = {
    "filter_diff_pressure_bar": {"name_zh": "濾網壓差",       "unit": "bar"},
    "servo_torque_load_pct":    {"name_zh": "伺服馬達負載",   "unit": "%"},
    "air_pressure_bar":         {"name_zh": "空氣壓力",       "unit": "bar"},
    "paint_flow_ml_min":        {"name_zh": "塗料流量",       "unit": "ml/min"},
    "spray_width_mm":           {"name_zh": "噴幅寬度",       "unit": "mm"},
    "path_error_mm":            {"name_zh": "軌跡追蹤誤差",   "unit": "mm"},
    "film_thickness_um":        {"name_zh": "膜厚",           "unit": "μm"},
    "vibration_g":              {"name_zh": "振動加速度",     "unit": "G"},
    "pump_current_a":           {"name_zh": "幫浦電流",       "unit": "A"},
    "gearbox_temperature_c":    {"name_zh": "減速機溫度",     "unit": "°C"},
    "temperature_c":            {"name_zh": "環境溫度",       "unit": "°C"},
    "humidity_rh":              {"name_zh": "環境濕度",       "unit": "%RH"},
}

# ── 負責單位中文標籤 ──
SKILL_ZH = {
    "operator":   "作業員",
    "technician": "技術員",
    "engineer":   "工程師",
}

# ── 狀態中文標籤 ──
STATE_ZH = {
    "warning": "警告",
    "fault":   "嚴重故障",
}


def build_alert_display_json(conn, event_id: str) -> dict:
    """
    前端告警顯示 JSON 建構函式（組合型，非單純 Query wrapper）

    目的
    ----
    讓前端可直接渲染告警面板，無需額外拼接多表資料。
    整合：問題原因、建議解法、負責單位、停機時間預估。

    Parameters
    ----------
    conn     : psycopg2 connection
    event_id : str — alert_event.event_id (UUID)

    Returns
    -------
    dict — 前端可直接使用的告警顯示 JSON

    Raises
    ------
    ValueError — event_id 不存在時
    """

    # ── Step 1：取 alert_event 基本資料 ──
    with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
        cur.execute(
            "SELECT * FROM alert_event WHERE event_id = %s;",
            (event_id,)
        )
        event = cur.fetchone()
    if event is None:
        raise ValueError(f"event_id 不存在：{event_id}")
    event = dict(event)

    # ── Step 2：查 alert_cause_link + cause_catalog ──
    with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
        cur.execute(
            """
            SELECT acl.cause_id, acl.is_primary,
                   cc.description_zh, cc.category, cc.severity
            FROM alert_cause_link acl
            JOIN cause_catalog cc ON acl.cause_id = cc.cause_id
            WHERE acl.alert_id = %s
            ORDER BY acl.is_primary DESC;
            """,
            (event_id,)
        )
        cause_rows = [dict(r) for r in cur.fetchall()]

    # ── Step 3：查 alert_response_link + response_catalog ──
    with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
        cur.execute(
            """
            SELECT arl.response_id, arl.executed_at, arl.operator_id,
                   rc.description_zh, rc.downtime_estimate_min, rc.skill_required
            FROM alert_response_link arl
            JOIN response_catalog rc ON arl.response_id = rc.response_id
            WHERE arl.alert_id = %s;
            """,
            (event_id,)
        )
        response_rows = [dict(r) for r in cur.fetchall()]

    # ── Step 4：查 cause_catalog 作為 fallback（若 cause_link 為空）──
    # alert_event.cause 欄位（VARCHAR）可能存有 cause_id 字串
    if not cause_rows and event.get("cause"):
        with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
            cur.execute(
                "SELECT cause_id, description_zh, category, severity FROM cause_catalog WHERE cause_id = %s;",
                (event["cause"],)
            )
            row = cur.fetchone()
            if row:
                cause_rows = [dict(row) | {"is_primary": True}]

    # ── Step 5：組合顯示 JSON ──
    sensor_name   = event["sensor_name"]
    display_info  = SENSOR_DISPLAY.get(sensor_name, {"name_zh": sensor_name, "unit": ""})
    state         = event["state"]
    state_zh      = STATE_ZH.get(state, state)
    measured_val  = event["measured_value"]
    unit          = display_info["unit"]

    # ── 問題摘要文字 ──
    problem_summary = (
        f"{event['station_id']} {display_info['name_zh']} "
        f"偵測到 {state_zh}（測量值：{measured_val} {unit}）"
    )

    causes = [
        {
            "cause_id":    c["cause_id"],
            "description": c["description_zh"],
            "category":    c.get("category"),
            "severity":    c["severity"],
            "is_primary":  bool(c.get("is_primary", False)),
        }
        for c in cause_rows
    ]

    actions = [
        {
            "response_id":           r["response_id"],
            "description":           r["description_zh"],
            "responsible_unit_code": r["skill_required"],
            "responsible_unit_zh":   SKILL_ZH.get(r["skill_required"], r["skill_required"]),
            "downtime_estimate_min": r["downtime_estimate_min"],
            "downtime_display":      f"約 {r['downtime_estimate_min']} 分鐘"
                                     if r["downtime_estimate_min"] else "不需停機",
            "is_executed":           r["executed_at"] is not None,
            "executed_at":           str(r["executed_at"]) if r["executed_at"] else None,
            "executed_by":           r["operator_id"],
        }
        for r in response_rows
    ]

    return {
        "event_id":           str(event["event_id"]),
        "station_id":         event["station_id"],
        "sensor_name":        sensor_name,
        "sensor_display_name": display_info["name_zh"],
        "measured_value":     float(measured_val),
        "unit":               unit,
        "state":              state,
        "state_zh":           state_zh,
        "ts":                 str(event["ts"]),
        "batch_id":           event["batch_id"],
        "acknowledged":       event["acknowledged_at"] is not None,
        "problem": {
            "summary": problem_summary,
            "causes":  causes,
        },
        "actions":    actions,
        "generated_at": datetime.now(timezone.utc).isoformat(),
    }


print("✅ 5. build_alert_display_json 定義完成")

---
## 6. 完整使用範例

以下範例前提：`setup_db.sql` 已執行，已有 demo / DataPreprocess 資料寫入 DB。

In [ ]:
conn = get_conn()

# ── 範例 6-A：DataPreprocess JSON 整包寫入 ──
report = load_json_to_db(conn, SAMPLE_PAYLOAD)
print("寫入報告：")
print(json.dumps(report, indent=2, ensure_ascii=False))

In [ ]:
# ── 範例 6-B：手動寫入告警事件（含 cause + response 連結）──
result = insert_alert_event(
    conn,
    batch_id       = "B_20260609_001",
    station_id     = "Station_1",
    sensor_name    = "filter_diff_pressure_bar",
    measured_value = 0.65,
    state          = "fault",
    cause_id       = "FILTER_CLOG",
    response_id    = "REPLACE_FILTER",
    ts             = "2026-06-09T08:30:00+08:00",
    message        = "濾網壓差超過 fault 門檻（0.7 bar），偵測值：0.65 bar",
    operator_id    = "operator_001"
)
print("insert_alert_event 結果：")
print(json.dumps(result, indent=2, ensure_ascii=False))

created_event_id = result["event_id"]
print(f"\n新建 event_id：{created_event_id}")

In [ ]:
# ── 範例 6-C：build_alert_display_json（前端告警顯示 JSON）──
display_json = build_alert_display_json(conn, event_id=created_event_id)
print(json.dumps(display_json, indent=2, ensure_ascii=False, default=str))

In [ ]:
# ── 範例 6-D：查詢 Station_1 時間序列 ──
df_ts = get_time_series(
    conn, station="Station_1",
    start_time="2026-06-01T00:00:00+08:00",
    end_time="2026-06-09T23:59:59+08:00",
    sensor_names=["filter_diff_pressure_bar", "servo_torque_load_pct", "air_pressure_bar"]
)
print(f"查詢筆數：{len(df_ts)}")
df_ts.head(5)

In [ ]:
# ── 範例 6-E：最新感測快照 ──
snapshot = get_latest_sensor_snapshot(conn, station="Station_1")
print(json.dumps({k: str(v) for k, v in snapshot.items()}, indent=2, ensure_ascii=False))

In [ ]:
# ── 範例 6-F：管理者儀表板摘要 ──
summary = get_manager_summary(conn, date="2026-06-09")
for k, v in summary.items():
    if k != "station_breakdown":
        print(f"{k}: {v}")
print("\n站點分布：")
pd.DataFrame(summary["station_breakdown"])

In [ ]:
# ── 範例 6-G：未確認告警（高嚴重度）──
df_alerts = list_unacknowledged_alert_events(conn, severity="high")
print(f"未確認高嚴重度告警：{len(df_alerts)} 筆")
if not df_alerts.empty:
    df_alerts[["event_id", "station_id", "sensor_name", "measured_value", "state", "ts"]].head()

In [ ]:
# ── 範例 6-H：濾網元件解方矩陣 ──
df_matrix = get_troubleshooting_matrix(conn, asset_type="FILTER")
df_matrix[["component_name", "issue_name", "solution_description",
           "relevance_rank", "effectiveness_pct", "skill_required"]]

In [ ]:
# ── 範例 6-I：批次清單（Station_1，2026-06-09）──
df_batches = list_batches_filtered(conn, date="2026-06-09", station="Station_1")
print(f"批次數量：{len(df_batches)}")
if not df_batches.empty:
    df_batches[["batch_id", "start_time", "ended_time", "status",
                "filter_state", "robot_arm_state", "quality_state"]].head()

In [ ]:
conn.close()
print("DB 連線已關閉")

---
## 函式總覽

| 類別 | 函式 | 說明 |
|---|---|---|
| **寫入** | `insert_batch_run` | batch_completed → batch_run |
| **寫入** | `insert_timeseries_to_sensor_1min` | time_series 1Hz PdM → sensor_1min |
| **寫入** | `insert_batch_sensor_to_sensor_1min` | 批次平均值 → sensor_1min |
| **寫入** | `insert_3min_env_to_sensor_3min` | 每 3 分鐘環境均值 → sensor_3min |
| **寫入** | `insert_alert_event` | 寫入告警 + 可選 cause / response 連結 |
| **寫入（主入口）** | `load_json_to_db` | 整包 DataPreprocess JSON → 各資料表 |
| **TimeSeriesService** | `get_time_series` | 時段感測資料（支援欄位篩選） |
| **TimeSeriesService** | `get_latest_sensor_snapshot` | 站點最新感測快照 |
| **TimeSeriesService** | `get_sensor_feature_window` | 滾動聚合特徵（avg / stddev / 超標次數） |
| **BatchService** | `list_batches_filtered` | 批次清單篩選 |
| **BatchService** | `get_batch_detail` | 單一批次完整資訊 |
| **DashboardService** | `get_manager_summary` | 管理者儀表板摘要 |
| **EventRuleService** | `list_unacknowledged_alert_events` | 未確認告警清單 |
| **EventRuleService** | `acknowledge_alert_event` | 確認告警 |
| **EventRuleService** | `evaluate_event_rules` | 批次評估門檻 + 自動寫入告警 |
| **TroubleshootingService** | `get_troubleshooting_matrix` | 元件－問題－解方矩陣 |
| **TroubleshootingService** | `get_issue_recommendations` | 特定問題建議解方 |
| **前端顯示（非 Query）** | `build_alert_display_json` | 整合原因+解法+負責單位+停機估計 → 前端 JSON |